In [54]:
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import recall_score
df_model = pd.read_csv(r"C:\Users\aashu\OneDrive\Desktop\Churn Prediction\data\raw_churn_data.csv")

In [55]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [56]:
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model.loc[df_model['tenure'] == 0, 'TotalCharges'] = 0
X, y = df_model.drop(columns=['Churn', 'customerID']), df_model['Churn']
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
print(num_cols ,"\n")
print(cat_cols)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object') 

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [57]:
# look at the sklearn ColumnTransformer or Pipeline documentation for review
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('ohe', (OneHotEncoder(handle_unknown='ignore')))])


preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

In [58]:
# X,y were identified in cell 2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Second split: 75% train - 25% validation for XGBoost early stopping (60% train, 20% val, 20% test overall)
X_train_xgb, X_val, y_train_xgb, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

#Using LogisticRegressionCV for AUTO hyperparameter tuning with *Stratified K-Folds* CV instead of KFold because its better for classification
logreg_pipe = Pipeline([('preprocess', preprocessor), ('logregcv', LogisticRegressionCV(max_iter=2000, scoring='recall'))])

logreg_pipe.fit(X_train, y_train)

y_pred = logreg_pipe.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix
print("Logistic Regression CV\n")
print('Best C:', logreg_pipe.named_steps['logregcv'].C_, '\n')
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Logistic Regression CV

Best C: [21.5443469] 

Confusion Matrix:
 [[925 110]
 [167 207]]

Classification Report:
               precision    recall  f1-score   support

          No       0.85      0.89      0.87      1035
         Yes       0.65      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409



In [59]:
from sklearn.ensemble import RandomForestClassifier


rf_pipe = Pipeline([('preprocess', preprocessor), ('randomforest', RandomForestClassifier(n_estimators= 300, oob_score= True, random_state=42, class_weight='balanced', n_jobs=-1))])

rf_pipe.fit(X_train, y_train)
y_pred = rf_pipe.predict(X_test)
print('Random Forest\n')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Random Forest

[[927 108]
 [197 177]]
              precision    recall  f1-score   support

          No       0.82      0.90      0.86      1035
         Yes       0.62      0.47      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [64]:
# Made yes and no numeric for xgboost
y_train_xgb_numeric = y_train_xgb.map({'No': 0, 'Yes': 1})
y_val_numeric = y_val.map({'No': 0, 'Yes': 1})
y_test_numeric = y_test.map({'No': 0, 'Yes': 1})

# Fit preprocessor ONLY on XGBoost training data, then transform all sets
preprocessor_xgb = clone(preprocessor)
preprocessor_xgb.fit(X_train_xgb)

X_train_processed = preprocessor_xgb.transform(X_train_xgb)
X_val_processed = preprocessor_xgb.transform(X_val)
X_test_processed = preprocessor_xgb.transform(X_test)

# XGBoost has early stopping to prevent overfitting to training set
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=300,
    random_state=42,
    scale_pos_weight=(y_train_xgb == 'No').sum() / (y_train_xgb == 'Yes').sum(),
    early_stopping_rounds=10
)

xgb.fit(
    X_train_processed, 
    y_train_xgb_numeric,  
    eval_set=[(X_val_processed, y_val_numeric)],
    verbose=False
)

y_pred = xgb.predict(X_test_processed)
print(f'XGBoost\n')
print(f"Best iteration: {xgb.best_iteration} (stopped at {xgb.best_iteration} out of 300)\n")
print("Confusion Matrix:\n", confusion_matrix(y_test_numeric, y_pred))
print("\nClassification Report:\n", classification_report(y_test_numeric, y_pred))

XGBoost

Best iteration: 15 (stopped at 15 out of 300)

Confusion Matrix:
 [[780 255]
 [100 274]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.75      0.81      1035
           1       0.52      0.73      0.61       374

    accuracy                           0.75      1409
   macro avg       0.70      0.74      0.71      1409
weighted avg       0.79      0.75      0.76      1409



In [65]:
print("LR Recall:", recall_score(y_test, logreg_pipe.predict(X_test), pos_label='Yes'))
print("RF Recall:", recall_score(y_test, rf_pipe.predict(X_test), pos_label='Yes'))
print("XGB Recall:", recall_score(y_test_numeric, xgb.predict(X_test_processed), pos_label=1))

LR Recall: 0.553475935828877
RF Recall: 0.4732620320855615
XGB Recall: 0.732620320855615
